<a href="https://colab.research.google.com/github/LusciousRCChigwena/Luscious-R-C-Chigwena-R2420728/blob/main/Financial_Econometrics_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Financial Econometrics Handbook: Volatility Modeling for a Derivatives Desk

This notebook is a modern, executable project designed for quant teams building volatility-aware derivative pricing and hedging guidance. It addresses four critical econometric challenges through definitions, examples, visualizations, diagnostics, damage analysis, and recommended directions.

## Project Scope

Selected challenges:
1. Multicollinearity
2. Skewness
3. Overfitting
4. Sensitivity to Outliers

## Executive Summary

This document presents a quant-driven handbook that combines live financial data, statistical diagnostics, and modern model guidance. The goal is to help traders and risk managers identify and resolve structural problems that can degrade volatility modeling and derivative pricing.

In [ ]:
# Install required packages if needed
import sys
import subprocess
import importlib
packages = ['numpy', 'pandas', 'matplotlib', 'seaborn', 'statsmodels', 'sklearn', 'scipy']
for pkg in packages:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# yfinance is optional; use live data if available
try:
    import yfinance as yf
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'yfinance'])
    import yfinance as yf

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

In [ ]:
# Load real-world financial data with fallback to simulation
def load_financial_data():
    try:
        tickers = ['SPY', 'QQQ', 'IWM', '^VIX']
        raw = None
        for attempt in range(3):
            try:
                raw = yf.download(
                    tickers,
                    start='2019-01-01',
                    end='2025-01-01',
                    progress=False,
                    threads=False,
                )
                break
            except Exception as exc:
                if attempt == 2:
                    raise
                time.sleep(2)

        if raw is None or raw.empty:
            raise RuntimeError('yfinance returned no data')

        if isinstance(raw.columns, pd.MultiIndex):
            raw = raw['Adj Close']
        elif 'Adj Close' in raw.columns:
            raw = raw[['Adj Close']]

        if raw.shape[1] == 4:
            raw.columns = ['SPY', 'QQQ', 'IWM', 'VIX']
        else:
            raw = raw[['SPY', 'QQQ', 'IWM', '^VIX']]
            raw.columns = ['SPY', 'QQQ', 'IWM', 'VIX']

        raw = raw.dropna()
        raw['SPY_return'] = raw['SPY'].pct_change()
        raw['QQQ_return'] = raw['QQQ'].pct_change()
        raw['IWM_return'] = raw['IWM'].pct_change()
        raw['VIX_change'] = raw['VIX'].diff()
        raw['SPY_vol'] = raw['SPY_return'].rolling(20).std() * np.sqrt(252)
        raw['QQQ_vol'] = raw['QQQ_return'].rolling(20).std() * np.sqrt(252)
        raw['IWM_vol'] = raw['IWM_return'].rolling(20).std() * np.sqrt(252)
        data = raw.dropna().copy()
        data['target'] = data['VIX'].shift(-1)
        return data
    except Exception as exc:
        print(f'Failed to download live data ({type(exc).__name__}): {exc}. Using simulated fallback data.')
        np.random.seed(42)
        dates = pd.date_range('2020-01-01', periods=800, freq='B')
        base = np.cumsum(np.random.normal(0, 0.001, size=len(dates))) + 3.7
        data = pd.DataFrame({
            'SPY': 400 + 20 * np.sin(np.linspace(0, 8, len(dates))) + base * 10,
            'QQQ': 300 + 15 * np.cos(np.linspace(0, 8, len(dates))) + base * 8,
            'IWM': 200 + 12 * np.sin(np.linspace(0, 5, len(dates))) + base * 5,
            'VIX': 18 + 3 * np.sin(np.linspace(0, 12, len(dates))) + np.random.normal(0, 0.5, len(dates))
        }, index=dates)
        data['SPY_return'] = data['SPY'].pct_change()
        data['QQQ_return'] = data['QQQ'].pct_change()
        data['IWM_return'] = data['IWM'].pct_change()
        data['VIX_change'] = data['VIX'].diff()
        data['SPY_vol'] = data['SPY_return'].rolling(20).std() * np.sqrt(252)
        data['QQQ_vol'] = data['QQQ_return'].rolling(20).std() * np.sqrt(252)
        data['IWM_vol'] = data['IWM_return'].rolling(20).std() * np.sqrt(252)
        data = data.dropna()
        data['target'] = data['VIX'].shift(-1)
        return data

financial_data = load_financial_data()
financial_data.head()

## Data Overview

The dataset includes liquid ETF prices (`SPY`, `QQQ`, `IWM`) and the implied volatility index (`VIX`). We derive daily returns and rolling volatility metrics to demonstrate econometric challenges in volatility modeling.

In [ ]:
financial_data[['SPY', 'QQQ', 'IWM', 'VIX', 'SPY_vol', 'QQQ_vol', 'IWM_vol']].tail()

---
## 1. Multicollinearity

### Definition
Multicollinearity occurs when two or more predictor variables in a regression model are highly linearly related. In matrix form, if X is the design matrix, the variance of OLS estimates depends on the inverse of X^T X, which becomes unstable when predictors are nearly collinear.

### Formula
Variance inflation factor (VIF) for predictor j is defined as:  VIF_j = 1 / (1 - R_j^2), where R_j^2 is the R-squared from regressing X_j on the remaining predictors.

### Description
Highly correlated financial factors can make coefficient estimates unstable and reduce the confidence of hedging signals. Even when the model appears precise, the estimated exposures can flip dramatically with small changes in data.

In [ ]:
features = financial_data[['SPY_vol', 'QQQ_vol', 'IWM_vol', 'SPY_return', 'QQQ_return', 'IWM_return']].copy()
corr = features.corr()
corr

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, linewidths=0.5)
plt.title('Correlation Matrix: Volatility and Return Predictors')
plt.show()

In [ ]:
def compute_vif(df):
    X = df.copy()
    X = X.dropna()
    vif_data = pd.DataFrame()
    vif_data['feature'] = X.columns
    vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    return vif_data

vif_results = compute_vif(features)
vif_results

### Diagnosis
Use a correlation heatmap and VIF table to identify predictor redundancy. A VIF greater than 5–10 is a strong sign the model is vulnerable to multicollinearity.

### Damage
Multicollinearity inflates variance in coefficient estimates, making volatility forecasts and hedging ratios unreliable. This can lead traders to over- or under-hedge option portfolios based on unstable factor combinations.

### Directions
Suggested remedies include:
- Using regularized regression such as Ridge to shrink correlated coefficients.
- Applying principal component analysis (PCA) or factor extraction to summarize redundant signals.
- Removing or combining predictors that carry the same economic information.

In [ ]:
X = features[['SPY_vol', 'QQQ_vol', 'IWM_vol']]
y = financial_data['target']
mask = X.notna().all(axis=1) & y.notna()
X = X[mask]
y = y[mask]
ridge = Ridge(alpha=5.0)
ridge.fit(X, y)
pd.DataFrame({'Predictor': X.columns, 'Ridge Coefficient': ridge.coef_})

---
## 2. Skewness

### Definition
Skewness measures the asymmetry of a distribution around its mean. For a random variable X, the sample skewness is often written as g1 = (1/n) um ((X_i - ar{X})^3) / s^3, where s is the sample standard deviation.

### Description
Return and volatility data frequently display negative skew: extreme losses are more common than extreme gains. This violates normality assumptions used by standard risk models and option pricing heuristics.

In [ ]:
skew_values = financial_data[['SPY_return', 'QQQ_return', 'IWM_return', 'VIX_change']].skew()
skew_values

In [ ]:
plt.figure(figsize=(14, 5))
ax = plt.subplot(1, 2, 1)
sns.histplot(financial_data['SPY_return'].dropna(), bins=50, kde=True, color='tab:blue')
ax.set_title('SPY Return Distribution')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')

ax = plt.subplot(1, 2, 2)
stats.probplot(financial_data['SPY_return'].dropna(), dist='norm', plot=plt)
plt.title('SPY Return QQ-Plot')
plt.tight_layout()
plt.show()

### Diagnosis
Use skewness statistics and QQ-plots to detect departures from the Gaussian assumption. Pronounced tails or systematic curvature indicates that standard volatility metrics may understate extreme risk.

### Damage
Ignoring skewness leads to risk models that underestimate downside exposure, which can result in insufficient option pricing reserves and poor hedging performance during market stress.

### Directions
Recommended approaches include:
- Modeling returns with skewed distributions such as the skew-normal or generalized hyperbolic family.
- Using risk measures that are sensitive to tail behavior, such as CVaR or historical stress returns.
- Applying transformations or robust volatility estimators to mitigate bias from extreme observations.

---
## 3. Overfitting

### Definition
Overfitting occurs when a model learns idiosyncratic noise rather than the underlying signal. In-sample error decreases while out-of-sample error increases. A classic sign is a large gap between training and validation performance.

### Description
Complex volatility models may appear very accurate on historical data but fail to generalize. Overfitting is especially damaging for trading desks because it can produce misleading hedge ratios and risk forecasts.

In [ ]:
demo_data = financial_data[['SPY_vol', 'SPY_return']].dropna().copy()
demo_data['SPY_vol_lag'] = demo_data['SPY_vol'].shift(1)
demo_data = demo_data.dropna()
X = demo_data[['SPY_vol_lag']]
y = demo_data['SPY_return']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, shuffle=False)
scores = []
for degree in range(1, 7):
    poly = PolynomialFeatures(degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train_poly)))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test_poly)))
    scores.append((degree, train_rmse, test_rmse))
pd.DataFrame(scores, columns=['Degree', 'Train RMSE', 'Test RMSE'])

In [ ]:
df_scores = pd.DataFrame(scores, columns=['Degree', 'Train RMSE', 'Test RMSE'])
plt.plot(df_scores['Degree'], df_scores['Train RMSE'], marker='o', label='Train RMSE')
plt.plot(df_scores['Degree'], df_scores['Test RMSE'], marker='o', label='Test RMSE')
plt.xlabel('Polynomial Degree')
plt.ylabel('Root MSE')
plt.title('Model Complexity vs. Out-of-Sample Performance')
plt.legend()
plt.show()

### Diagnosis
Compare training and validation metrics for models of varying complexity. A model that continues to improve in-sample while validation error worsens is overfitting. Time-series-aware splits and cross-validation strengthen this diagnosis.

### Damage
Overfitting creates false confidence in volatility forecasts. When markets change, the desk may follow an unstable strategy that performs poorly in real trading, leading to losses and hedging mismatches.

### Directions
To prevent overfitting:
- Use simpler models and prefer parsimonious factor sets.
- Apply regularization such as Ridge or Lasso.
- Validate with out-of-sample periods and rolling cross-validation.

In [ ]:
alpha_values = [0.1, 1.0, 10.0, 100.0]
regularization_results = []
for alpha in alpha_values:
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    regularization_results.append({'alpha': alpha, 'Train RMSE': np.sqrt(mean_squared_error(y_train, model.predict(X_train))), 'Test RMSE': np.sqrt(mean_squared_error(y_test, model.predict(X_test)))})
pd.DataFrame(regularization_results)

---
## 4. Sensitivity to Outliers

### Definition
Outliers are observations that differ markedly from the majority of the data. In regression, they can exert undue influence on parameter estimates and distort the fitted model.

### Description
A small number of extreme price moves can dominate estimates of volatility and risk exposures. If the model is not robust, these anomalies can mislead hedging and capital decisions.

In [ ]:
demo_outlier = financial_data[['SPY_vol', 'SPY_return']].dropna().copy()
demo_outlier['SPY_vol_lag'] = demo_outlier['SPY_vol'].shift(1)
demo_outlier = demo_outlier.dropna().reset_index(drop=True)
# Introduce a synthetic outlier in the response
demo_outlier.loc[10, 'SPY_return'] = demo_outlier['SPY_return'].mean() + 12 * demo_outlier['SPY_return'].std()
X_all = demo_outlier[['SPY_vol_lag']]
y_all = demo_outlier['SPY_return']
ols = LinearRegression()
ols.fit(X_all, y_all)
sample_preds = ols.predict(X_all)
plt.scatter(X_all, y_all, label='Observations', alpha=0.7)
plt.plot(X_all, sample_preds, color='red', linewidth=2, label='OLS Fit')
plt.scatter(X_all.loc[10], y_all.loc[10], color='black', s=120, marker='X', label='Injected Outlier')
plt.xlabel('Lagged SPY Volatility')
plt.ylabel('SPY Return')
plt.title('Outlier Sensitivity in Linear Fit')
plt.legend()
plt.show()

### Diagnosis
Use residual plots, Cook's distance, and leverage statistics to identify influential observations. In time-series data, inspect extreme shocks and confirm whether they are data errors, regime shifts, or true market stress.

### Damage
Outliers can skew volatility surfaces and lead to mis-priced options or improper hedge ratios. This is especially costly when risk managers mistakenly treat a stress event as normal behavior.

### Directions
Robust modeling practices include:
- Using robust regression or quantile regression to reduce sensitivity to extreme observations.
- Trimming or winsorizing the data for exploratory model building.
- Monitoring shocks separately and treating them as regime-change events in model design.

---
## Practical Recommendations for the Derivatives Desk

1. Build a model pipeline that starts with feature decorrelation or dimensionality reduction.
2. Validate skew and tail behavior before relying on normal-based volatility signals.
3. Keep volatility forecasting models parsimonious and prefer regularization when many factors are available.
4. Treat outliers as potential regime signals, not just noise: use robust estimators and stress-specific models.

## References

- Hull, John C. *Options, Futures, and Other Derivatives*. 10th ed., Pearson, 2017.
- Andersen, Torben G., and Tim Bollerslev.
 *International Economic Review*, vol. 38, no. 4, 1997, pp. 885–905.
- Hendry, David F. *Dynamic Econometrics*. Oxford University Press, 1995.